# 📊 Notebook 01: 电力数据获取

## 学习目标
1. 理解电力数据的来源和种类
2. 学会用 Python 自动拉取中国电力数据
3. 掌握 DataFrame 的基本操作和可视化

## 背景知识

### 电力数据的层级
电力数据按时间粒度分为:
- **年级 (Yearly)**: 年度发电量、用电量，适合宏观趋势分析
- **月级 (Monthly)**: 月度供需数据，适合季节性分析
- **日级 (Daily)**: 每日负荷曲线，开始可以做时序预测
- **小时级 (Hourly)**: 小时级负荷数据，可以做短期负荷预测

### 我们的数据源
我们使用 **Our World in Data (OWID)** 提供的中国电力年度数据。
OWID 是一个完全开源的全球数据库，整合了 Ember、EIA、IEA 等多方权威来源。

**为什么选 OWID？**
1. 🆓 完全免费、开源
2. 🔄 自动更新 — GitHub 上的 CSV 文件
3. 🇨🇳 包含中国数据 (iso_code=CHN)，2000-2025
4. 📡 一行代码就能拉取 — 不需要 API Key

> 💡 **思考**: 为什么不直接用中国政府的开放数据平台？
> 中国地方政府的电力数据平台（如国家统计局、菏泽开放平台）
> 设置了反爬机制（验证码、JS 渲染、TLS 限制），无法程序化自动获取。
> 这就是为什么我们需要 DataLoader 抽象层——
> 当手动获取到日/小时数据后，可以无缝切换。

In [ ]:
# ── 导入依赖 ──────────────────────────────────
import sys
sys.path.insert(0, '..')  # 让 Python 能找到 pipeline/ 模块

from pipeline.data_loader import create_loader
from pipeline.cleaner import clean_data, validate_schema
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'  # 在 Jupyter 中渲染交互图表

---
## 步骤 1: 拉取中国电力数据

调用 `create_loader('owid')` 创建一个 OWID 数据加载器，
然后 `.load_data()` 自动从 GitHub 拉取 CSV、过滤中国数据、标准化格式。

### 数据加载器的工作原理
```
create_loader('owid')
  → OWIDChinaLoader.__init__()
  → load_data()
    → urllib 流式读取 25MB CSV
    → csv.DictReader 逐行解析
    → 过滤 iso_code='CHN' + year>=2000
    → TWh → MW 转换
    → 返回标准化 DataFrame
```

In [ ]:
# 创建 OWID 加载器，拉取中国电力数据
loader = create_loader('owid')
df_raw = loader.load_data()

# 查看数据概况
print(f"数据形状: {df_raw.shape[0]} 行 × {df_raw.shape[1]} 列")
print(f"时间范围: {df_raw['timestamp'].min().year} - {df_raw['timestamp'].max().year}")
df_raw.head(8)

### 📖 数据列说明

| 列名 | 含义 | 单位 |
|------|------|------|
| `timestamp` | 时间戳 (年中日期作为代表) | UTC |
| `load_mw` | 日均用电负荷 | MW (兆瓦) |
| `generation_twh` | 年发电量 | TWh (太瓦时) |
| `demand_twh` | 年用电量 | TWh |
| `solar_twh` | 太阳能发电量 | TWh |
| `wind_twh` | 风力发电量 | TWh |
| `coal_twh` | 煤电发电量 | TWh |

**TWh → MW 是怎么换算的？**
- 1 TWh(太瓦时) = 1,000,000 MWh(兆瓦时)
- 日均负荷 = TWh × 1,000,000 ÷ 365 天
- 例: 2024年用电量 ≈ 10,070 TWh → 日均 ≈ 27,589 MW
- 这说明全中国平均每时每刻消耗约 2,759 万千瓦的电力 ⚡

---
## 步骤 2: 数据清洗

`clean_data()` 执行以下操作:
1. ✅ 验证必需列存在
2. 📊 IQR 异常值检测（只报告，不删除）
3. 🕐 时区标准化为 UTC

In [ ]:
df = clean_data(df_raw)

# 验证清洗结果
result = validate_schema(df)
print(f"Schema 验证: {'✅ 通过' if result['valid'] else '❌ 失败'}")
if result['issues']:
    for issue in result['issues']:
        print(f"  ! {issue}")
print(f"\n数据概况:")
print(f"  行数: {result['summary']['rows']}")
print(f"  负荷范围: {result['summary']['load_min']:.0f} - {result['summary']['load_max']:.0f} MW")
df.head()

---
## 步骤 3: 数据可视化

用交互式图表探索中国电力趋势。
Plotly 支持鼠标悬停查看精确数值、框选放大、双击重置。

In [ ]:
# ── 图表 1: 中国年度发电量趋势 ────────────────
fig = go.Figure()

# 堆叠面积图——展示发电结构的变化
fig.add_trace(go.Scatter(
    x=df['timestamp'], y=df['coal_twh'],
    name='煤电', mode='lines', stackgroup='one',
    line=dict(color='#8B4513', width=0.5),
    fillcolor='rgba(139, 69, 19, 0.4)'
))
fig.add_trace(go.Scatter(
    x=df['timestamp'], y=df['wind_twh'],
    name='风电', mode='lines', stackgroup='one',
    line=dict(color='#00CED1', width=0.5),
    fillcolor='rgba(0, 206, 209, 0.4)'
))
fig.add_trace(go.Scatter(
    x=df['timestamp'], y=df['solar_twh'],
    name='太阳能', mode='lines', stackgroup='one',
    line=dict(color='#FFD700', width=0.5),
    fillcolor='rgba(255, 215, 0, 0.4)'
))

fig.update_layout(
    title='中国发电结构变化 (2000-2025)',
    xaxis_title='年份', yaxis_title='发电量 (TWh)',
    hovermode='x unified', height=500
)
fig.show()

### 📖 从图表中你能看到什么？

1. **煤电仍占大头** — 但占比在下降（从 80%+ 到 ~60%）
2. **风光爆发式增长** — 2015 年后太阳能和风能几乎是指数增长
3. **总用电量持续增长** — 中国经济在增长，用电量也在增长

> 💡 **思考**: 如果你是电力交易员，
> 风光发电的快速增长意味着什么？
> → 电力供应越来越依赖天气，波动性增加
> → 需要更精准的预测模型来应对不确定性
> → 这正好是我们接下来要做的事情！

---
## 📝 学习笔记

- 我们成功从 OWID 拉取了中国 2000-2025 的电力数据
- 数据加载器设计为**抽象接口**，未来可以无缝切换到日级/小时级数据
- 数据清洗管道已经就绪（IQR 检测、时区标准化）

**下一步: Notebook 02 → 深入数据清洗**

### 思考题

1. OWID 提供的是年份级别的数据。如果想预测明天的负荷，年级数据够用吗？差在哪里？
2. DataLoader 为什么要设计为抽象基类？直接写一个函数读取 CSV 不行吗？
3. 为什么 _twh_to_daily_mw 除以 365？如果是小时数据，公式会变成什么样？

